## This notebook will serve as a way to implement character generation RNN

In [1]:
import torch
import re
import numpy as np

In [2]:
with open("corpus_RNN_Rohff.txt", "r", encoding="utf-8", errors="replace") as f:
    corpus = f.read()

As the task will be char generation, first I will let indications such as <INTRO>, etc... in order to maybe have a RNN that can reproduce the structure of a song

In [5]:
corpus[:200]

"<STROPHE>\n<INTRO>\n<STROPHE>\nT'as le swag du XV de France, ambiance Mir couleur, mets de l'huile\nFonce-dé sur scène en mode strip-teaser\nTu t'sers de nos codes, de nos mots, d'nos méthodes\nUn reporter "

We need first to clean the 

In [6]:
print(len(set(corpus)),sorted(set(corpus)))

124 ['\n', ' ', '!', '#', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}', '\x81', '\x9c', '«', '³', '»', '¿', 'À', 'Â', 'Ä', 'Ç', 'É', 'Ê', 'Î', 'à', 'á', 'â', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'î', 'ï', 'ô', 'ù', 'û', 'ü', 'ÿ', 'ā', 'œ', 'е', '‘', '’', '“', '”', '…', '€']


In [87]:
corpus = corpus.replace("‘",'"')
corpus= corpus.replace("’",'"')
corpus = corpus.replace("“", '"')
corpus = corpus.replace("”", '"')
corpus = corpus.replace("»", '"')
corpus = corpus.replace("«", '"')
corpus = corpus.replace("*", "")
corpus = corpus.replace("³","")
corpus = corpus.replace("…","...")
corpus = corpus.replace("¿","")
corpus = re.sub(r"\{.*?\}", "", corpus)

In [88]:
print(len(set(corpus)),sorted(set(corpus)))

73 ['\n', ' ', '!', '"', '#', '%', '&', "'", '+', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ';', '=', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\x81', '\x9c', 'à', 'á', 'â', 'ä', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'î', 'ï', 'ô', 'ù', 'û', 'ü', 'ÿ', 'ā', 'œ', 'е', '€']


In [98]:
corpus=corpus.replace("á","a")
corpus=corpus.replace("ä","a")
corpus=corpus.replace("ì","i")
corpus=corpus.replace("ÿ","y")
corpus=corpus.replace("е","e")
corpus=corpus.replace("\x81","")
corpus=corpus.replace("\x9c","")

In [99]:
print(len(set(corpus)),sorted(set(corpus)))

66 ['\n', ' ', '!', '"', '#', '%', '&', "'", '+', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ';', '=', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'à', 'â', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'î', 'ï', 'ô', 'ù', 'û', 'ü', 'ā', 'œ', '€']


## Now we can create a mapping of each character for encoding

In [100]:
chars_sort = sorted(set(corpus))

In [101]:
# encode chars
char2int = {ch: i for i, ch in enumerate(chars_sort)}

# `char2int` dictionary for char -> int
print(char2int)

{'\n': 0, ' ': 1, '!': 2, '"': 3, '#': 4, '%': 5, '&': 6, "'": 7, '+': 8, '-': 9, '.': 10, '0': 11, '1': 12, '2': 13, '3': 14, '4': 15, '5': 16, '6': 17, '7': 18, '8': 19, '9': 20, ';': 21, '=': 22, 'a': 23, 'b': 24, 'c': 25, 'd': 26, 'e': 27, 'f': 28, 'g': 29, 'h': 30, 'i': 31, 'j': 32, 'k': 33, 'l': 34, 'm': 35, 'n': 36, 'o': 37, 'p': 38, 'q': 39, 'r': 40, 's': 41, 't': 42, 'u': 43, 'v': 44, 'w': 45, 'x': 46, 'y': 47, 'z': 48, 'à': 49, 'â': 50, 'æ': 51, 'ç': 52, 'è': 53, 'é': 54, 'ê': 55, 'ë': 56, 'î': 57, 'ï': 58, 'ô': 59, 'ù': 60, 'û': 61, 'ü': 62, 'ā': 63, 'œ': 64, '€': 65}


In [136]:
int2char = {i: ch for ch, i in char2int.items()}

# `int2char` for int -> char
print(int2char)

{0: '\n', 1: ' ', 2: '!', 3: '"', 4: '#', 5: '%', 6: '&', 7: "'", 8: '+', 9: '-', 10: '.', 11: '0', 12: '1', 13: '2', 14: '3', 15: '4', 16: '5', 17: '6', 18: '7', 19: '8', 20: '9', 21: ';', 22: '=', 23: 'a', 24: 'b', 25: 'c', 26: 'd', 27: 'e', 28: 'f', 29: 'g', 30: 'h', 31: 'i', 32: 'j', 33: 'k', 34: 'l', 35: 'm', 36: 'n', 37: 'o', 38: 'p', 39: 'q', 40: 'r', 41: 's', 42: 't', 43: 'u', 44: 'v', 45: 'w', 46: 'x', 47: 'y', 48: 'z', 49: 'à', 50: 'â', 51: 'æ', 52: 'ç', 53: 'è', 54: 'é', 55: 'ê', 56: 'ë', 57: 'î', 58: 'ï', 59: 'ô', 60: 'ù', 61: 'û', 62: 'ü', 63: 'ā', 64: 'œ', 65: '€'}


In [108]:
corpus_encoded = np.array([char2int[ch] for ch in corpus])

In [116]:
n_letters = len(int2char)

In [125]:
def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters)
    for li, letter in enumerate(line):
        tensor[li][0][char2int[letter]] = 1
    return tensor

In [128]:
chars = sorted(list(set(corpus)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

encoded = [stoi[c] for c in corpus]

In [145]:
seq_length = 100

inputs = []
targets = []
all_datas = []

for i in range(0, len(corpus_encoded) - seq_length) :
    inputs.append(corpus_encoded[i:i+seq_length])
    targets.append(corpus_encoded[i+1:i+seq_length+1])
    all_datas.append([corpus_encoded[i:i+seq_length],corpus_encoded[i+1:i+seq_length+1]])


In [148]:
print([int2char[char] for char in inputs[0]])

print("".join([int2char[char] for char in inputs[0]]))

['\n', 't', "'", 'a', 's', ' ', 'l', 'e', ' ', 's', 'w', 'a', 'g', ' ', 'd', 'u', ' ', 'x', 'v', ' ', 'd', 'e', ' ', 'f', 'r', 'a', 'n', 'c', 'e', ' ', 'a', 'm', 'b', 'i', 'a', 'n', 'c', 'e', ' ', 'm', 'i', 'r', ' ', 'c', 'o', 'u', 'l', 'e', 'u', 'r', ' ', 'm', 'e', 't', 's', ' ', 'd', 'e', ' ', 'l', "'", 'h', 'u', 'i', 'l', 'e', '\n', 'f', 'o', 'n', 'c', 'e', '-', 'd', 'é', ' ', 's', 'u', 'r', ' ', 's', 'c', 'è', 'n', 'e', ' ', 'e', 'n', ' ', 'm', 'o', 'd', 'e', ' ', 's', 't', 'r', 'i', 'p', '-']

t'as le swag du xv de france ambiance mir couleur mets de l'huile
fonce-dé sur scène en mode strip-


In [ ]:
import torch
from torch.utils.data import Dataset


class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks

    def __len__(self):
        return len(self.text_chunks)

    def __getitem__(self, idx):
        x, y = self.text_chunks[idx]
        return torch.tensor(x), torch.tensor(y)  # return input, target

seq_dataset = TextDataset(torch.tensor(all_datas))

In [155]:
from torch.utils.data import DataLoader

batch_size = 64

torch.manual_seed(42)
seq_dl = DataLoader(seq_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [162]:
for i, (x,y) in enumerate(seq_dl) :
    print(x[0],y[0])
    break

tensor([26,  1, 27, 34, 34, 27,  1, 38, 37, 43, 41, 41, 27,  0, 40, 31, 27, 36,
         1, 39, 43,  7, 37, 36,  1, 41,  7, 38, 37, 43, 41, 41, 27,  1, 23, 43,
         1, 24, 37, 40, 26,  1, 26, 27,  1, 34, 49,  1, 47,  7, 23,  1, 38, 23,
        41,  1, 26, 27,  1, 38, 37, 43, 41, 41, 27, 41,  0, 26, 37, 36, 25,  1,
        29, 27, 34, 23,  1, 24, 23, 31, 41, 41, 27,  1, 32, 23, 35, 23, 31, 41,
         1, 34, 23,  1, 29, 23, 40, 26, 27,  1]) tensor([ 1, 27, 34, 34, 27,  1, 38, 37, 43, 41, 41, 27,  0, 40, 31, 27, 36,  1,
        39, 43,  7, 37, 36,  1, 41,  7, 38, 37, 43, 41, 41, 27,  1, 23, 43,  1,
        24, 37, 40, 26,  1, 26, 27,  1, 34, 49,  1, 47,  7, 23,  1, 38, 23, 41,
         1, 26, 27,  1, 38, 37, 43, 41, 41, 27, 41,  0, 26, 37, 36, 25,  1, 29,
        27, 34, 23,  1, 24, 23, 31, 41, 41, 27,  1, 32, 23, 35, 23, 31, 41,  1,
        34, 23,  1, 29, 23, 40, 26, 27,  1, 28])


C:\Users\HLM\AppData\Local\Temp\ipykernel_26144\3681504253.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(x), torch.tensor(y)  # return input, target


In [163]:
import torch.nn as nn

In [165]:
len(char2int)

66

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, num_layers, batch_first=True)        
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)              # (batch, seq, hidden_size)
        out, hidden = self.rnn(x, hidden)   
        out = self.fc(out)                  
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size)

In [233]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Example
vocab_size = 66
hidden_size = 128
batch_size = 64
seq_len = 100

model = CharRNN(vocab_size, hidden_size)
hidden = model.init_hidden(batch_size)

model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [230]:
pred[0][0]

tensor([ 1.5566,  1.6886,  2.0717,  1.7174, -4.2841, -3.1262, -0.6376,  3.6344,
        -5.4765,  0.1134, -4.7320, -2.4151, -1.0996, -0.4061, -4.4051, -4.4500,
        -5.0813, -3.2953, -2.7382, -3.0114, -4.0713, -4.4445, -4.8885,  4.0381,
        -2.5100, -2.6416, -4.1938,  5.2864,  1.5063, -2.5934,  0.7882,  3.2626,
        -2.3963, -0.0611,  2.4850, -0.3342,  0.1439,  3.7770, -3.5754, -5.7981,
         0.2027, -0.3954,  1.1055,  4.1173, -6.4602, -0.1384, -3.8318,  0.6540,
        -3.1024, -1.6358, -2.0420, -4.9152, -4.6977,  1.4412,  3.0307, -1.0143,
        -5.0947, -5.0517, -3.9874,  0.7546, -4.6917, -0.8323, -4.0808, -3.8179,
        -3.9592, -5.0053], grad_fn=<SelectBackward0>)

In [231]:
target_batch[0][0]

tensor(27)

In [ ]:
# for execution time measurement
from timeit import default_timer as timer

num_epochs = 5000
model.train()

start = timer()  # timer start
for epoch in range(num_epochs):
    hidden = model.init_hidden(batch_size)

    seq_batch, target_batch = next(iter(seq_dl))
    seq_batch = seq_batch.to(device)
    target_batch = target_batch.to(device)

    optimizer.zero_grad()
    loss = 0

    #for c in range(seq_length):
    pred, hidden = model(seq_batch, hidden)
    loss += loss_fn(pred.view(-1, vocab_size), target_batch.view(-1))

    loss.backward()
    optimizer.step()

    loss = loss.item() / seq_length
    if epoch % 1000 == 0:
        print(f"Epoch {epoch} loss: {loss:.4f}")

end = timer()  # timer end
print("Total execution time in seconds: ", "%.2f" % (end - start))
print("Device type: ", device)

C:\Users\HLM\AppData\Local\Temp\ipykernel_26144\3681504253.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(x), torch.tensor(y)  # return input, target


Epoch 0 loss: 0.0429
Epoch 1000 loss: 0.0173
Epoch 2000 loss: 0.0169
Epoch 3000 loss: 0.0163


KeyboardInterrupt: 

In [ ]:
for i, (x,y) in enumerate(seq_dl) :
    
    break

In [179]:
len(x)

64

In [177]:
len(hidden[0][0])

128

In [235]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Example
vocab_size = 66
hidden_size = 128
batch_size = 64
seq_len = 100

model = CharRNN(vocab_size, hidden_size, num_layers=4)
hidden = model.init_hidden(batch_size)

model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [236]:
# for execution time measurement
from timeit import default_timer as timer

num_epochs = 5000
model.train()

start = timer()  # timer start
for epoch in range(num_epochs):
    hidden = model.init_hidden(batch_size)

    seq_batch, target_batch = next(iter(seq_dl))
    seq_batch = seq_batch.to(device)
    target_batch = target_batch.to(device)

    optimizer.zero_grad()
    loss = 0

    #for c in range(seq_length):
    pred, hidden = model(seq_batch, hidden)
    loss += loss_fn(pred.view(-1, vocab_size), target_batch.view(-1))

    loss.backward()
    optimizer.step()

    loss = loss.item() / seq_length
    if epoch % 1000 == 0:
        print(f"Epoch {epoch} loss: {loss:.4f}")

end = timer()  # timer end
print("Total execution time in seconds: ", "%.2f" % (end - start))
print("Device type: ", device)

C:\Users\HLM\AppData\Local\Temp\ipykernel_26144\3681504253.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(x), torch.tensor(y)  # return input, target


Epoch 0 loss: 0.0415
Epoch 1000 loss: 0.0155
Epoch 2000 loss: 0.0149
Epoch 3000 loss: 0.0146
Epoch 4000 loss: 0.0149
Total execution time in seconds:  1540.69
Device type:  cpu
